# Speedy Analysis — a working tour of this `pandas_ta` fork

This notebook computes a few hundred indicator columns on one price series and then does the thing
that actually distinguishes this fork: **it checks them.**

Upstream `pandas-ta` gives you indicators. This fork treats every indicator as a candidate
**machine-learning feature**, which imposes three requirements a chart-drawing library never has to
meet — a column must be **causal**, **scale-free**, and **non-redundant**. Sections 5–7 test all
three, live, on the columns computed above them, and report the failures rather than hiding them.

**What you will see, in order**

| § | What |
|---|---|
| 1 | Loading data (real, or synthetic if you are offline) |
| 2 | The three call styles |
| 3 | The bulk indicator sweep — one spec table, every call checked |
| 4 | Multi-output indicators, consolidated into usable signals |
| 5 | What *this fork* adds that upstream does not have |
| 6 | The ML feature contract: causality, scale-freedom, redundancy — measured |
| 7 | Coverage, computed from the package rather than claimed |

**Where the boundary is written down:** [`docs/SHOWCASE_SCOPE.md`](../docs/SHOWCASE_SCOPE.md) records, block by block, what may appear in these examples and what may not. `tests/test_showcase.py` enforces it.

**What is deliberately not here:** no trading strategy, no thresholds, no scoring weights, no buy or
sell signal. This notebook shows what the library computes. What to *do* with those numbers is a
separate question and not one a library example should answer for you.

## 1. Setup and data

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import pandas_ta as ta

print(f"pandas_ta {ta.version}")
print(f"pandas    {pd.__version__}")
print(f"numpy     {np.__version__}")

pandas_ta 0.2.67b1+tu.1
pandas    2.3.3
numpy     1.26.4


The notebook runs **offline**. It tries Yahoo Finance first; if that fails for any reason — no
network, rate limit, no `yfinance` installed — it falls back to a synthetic geometric-random-walk
frame and says so. Every result below is reproducible either way.

In [2]:
TICKER = "AAPL"
PERIOD = "3y"


def synthetic_ohlcv(n=750, seed=7):
    """A geometric random walk with a coherent OHLC envelope."""
    rng = np.random.default_rng(seed)
    close = 100 * np.exp(np.cumsum(rng.normal(0, 0.012, n)))
    open_ = close * (1 + rng.normal(0, 0.003, n))
    high = np.maximum(open_, close) * (1 + abs(rng.normal(0, 0.006, n)))
    low = np.minimum(open_, close) * (1 - abs(rng.normal(0, 0.006, n)))
    return pd.DataFrame(
        {"open": open_, "high": high, "low": low, "close": close,
         "volume": rng.integers(1e5, 1e6, n).astype(float)},
        index=pd.date_range("2022-01-01", periods=n, freq="B"),
    )


def load(ticker=TICKER, period=PERIOD):
    try:
        import yfinance as yf
        raw = yf.download(ticker, period=period, auto_adjust=True,
                          progress=False, multi_level_index=False)
        if raw is None or raw.empty:
            raise RuntimeError("empty frame")
        raw.columns = [str(c).lower() for c in raw.columns]
        df = raw[["open", "high", "low", "close", "volume"]].dropna()
        if len(df) < 250:
            raise RuntimeError(f"only {len(df)} bars")
        return df, f"{ticker} via yfinance"
    except Exception as exc:
        return synthetic_ohlcv(), f"synthetic random walk (yfinance unavailable: {exc})"


df, SOURCE = load()
print(f"source : {SOURCE}")
print(f"bars   : {len(df)}  ({df.index[0].date()} to {df.index[-1].date()})")
df.tail(3)

source : AAPL via yfinance
bars   : 753  (2023-09-12 to 2026-09-11)


,open,high,low,close,volume
Date,,,,,
2026-09-09,315.489990,319.149994,309.899994,315.339996,65640000
2026-09-10,316.670013,326.739990,316.510010,326.570007,70011900
2026-09-11,327.450012,336.220001,326.299988,332.269989,50659000


> ### ⚠ This data is not cleaned
>
> The loader hands the frame over exactly as Yahoo returns it. `auto_adjust=True` applies Yahoo's
> own split and dividend adjustment **where Yahoo has a record of one** — it is not a
> data-integrity pass, and it is easy to read as one.
>
> Nothing here repairs an unadjusted corporate action, and nothing here checks a bar against an
> exchange's daily price limit. On an exchange that has one, an unadjusted split shows up as a
> single impossible bar, and every windowed indicator that touches it is poisoned for the length
> of its window — longer for anything with an EMA or a recursive stage, where the parameter is a
> decay constant rather than a window.
>
> Cleaning belongs in your data layer. An indicator library is the wrong place for it, and this
> notebook is not doing it for you.

Two conventions worth knowing before anything else, because both bite people:

**Lower-case column names.** `pandas_ta` expects `open/high/low/close/volume`. Yahoo returns them
capitalised, which is why the loader lower-cases them. Get this wrong and the `df.ta` accessor
silently finds nothing.

**Column names are the API.** `STOCHk_14_3_3` and `BBU_20_2.0` encode their parameters in the name.
That is not cosmetic — downstream code matches on those strings, so renaming a column is a breaking
change in this fork, treated the same as changing a function signature.

## 2. The three call styles

One implementation, three ways in. Pick whichever suits the code you are writing; they return the
same numbers.

In [3]:
# Style 1 — standard: pass Series explicitly. Most explicit, easiest to debug.
rsi_a = ta.rsi(df["close"], length=14)

# Style 2 — the DataFrame accessor: pandas_ta finds the columns by name.
rsi_b = df.ta.rsi(length=14)

# Style 3 — bulk, via Category. `df.ta.strategy()` sweeps whole categories at once.
#            Shown here but not run: it is multiprocessed and slow for a demo.

print(f"styles 1 and 2 identical: {rsi_a.equals(rsi_b)}")
print(f"column name             : {rsi_a.name}")
rsi_a.tail(3)

styles 1 and 2 identical: True
column name             : RSI_14


Date
2026-09-09    49.401551
2026-09-10    58.992827
2026-09-11    62.842819
Name: RSI_14, dtype: float64

## 3. The bulk sweep

A spec table — indicator name, which price series it takes, its parameters — driven in a loop. This
is the pattern most dashboards end up writing.

**The failure-reporting here is the point.** The original script this notebook is derived from
wrapped each call in a bare `except: continue`. That looks harmless and is not: **16 of its
indicator specs passed argument lists that did not match the function signature**, raised
`TypeError` on every call, and were swallowed silently. They had never produced a column, and
nothing in the output said so.

So this loop collects failures and prints them. A silent sweep is not a working sweep — it is an
untested one.

In [4]:
O, H, L, C, V = "open", "high", "low", "close", "volume"

SPECS = [
    # --- volatility / range -------------------------------------------------
    ("aberration", [H, L, C], {}),          ("accbands", [H, L, C], {}),
    ("atr",   [H, L, C], {"length": 14}),   ("natr", [H, L, C], {"length": 14}),
    ("true_range", [H, L, C], {}),          ("bbands", [C], {"length": 20, "std": 2}),
    ("kc",    [H, L, C], {"length": 20}),   ("donchian", [H, L], {}),
    ("hilo",  [H, L, C], {"high_length": 14, "low_length": 14}),
    ("massi", [H, L], {}),                  ("pdist", [O, H, L, C], {}),
    ("rvi",   [H, L, C], {}),               ("thermo", [H, L, C], {}),
    ("ui",    [C], {}),                     ("stdev", [C], {"length": 20}),
    ("variance", [C], {}),                  ("chop", [H, L, C], {}),
    # --- momentum / oscillators --------------------------------------------
    ("rsi",  [C], {"length": 14}),          ("rsx", [C], {}),
    ("cci",  [H, L, C], {"length": 20}),    ("cmo", [C], {"length": 14}),
    ("roc",  [C], {"length": 12}),          ("mom", [C], {"length": 10}),
    ("stoch", [H, L, C], {"k": 14, "d": 3}), ("stochrsi", [C], {"length": 14}),
    ("kdj",  [H, L, C], {}),                ("willr", [H, L, C], {"length": 14}),
    ("uo",   [H, L, C], {}),                ("ao", [H, L], {}),
    ("apo",  [C], {}),                      ("ppo", [C], {}),
    ("tsi",  [C], {}),                      ("smi", [C], {}),
    ("trix", [C], {"length": 14}),          ("kst", [C], {}),
    ("coppock", [C], {}),                   ("fisher", [H, L], {}),
    ("er",   [C], {}),                      ("cfo", [C], {}),
    ("cg",   [C], {}),                      ("bop", [O, H, L, C], {}),
    ("brar", [O, H, L, C], {}),             ("eri", [H, L, C], {}),
    ("inertia", [C], {}),                   ("pgo", [H, L, C], {}),
    ("psl",  [C], {}),                      ("qstick", [O, C], {}),
    ("rvgi", [O, H, L, C], {}),             ("squeeze", [H, L, C], {}),
    ("ebsw", [C], {}),                      ("bias", [C], {}),
    # --- trend ---------------------------------------------------------------
    ("adx",  [H, L, C], {"length": 14}),    ("amat", [C], {}),
    ("aroon", [H, L], {"length": 25}),      ("cksp", [H, L, C], {}),
    ("dpo",  [C], {"centered": False}),     ("increasing", [C], {}),
    ("decreasing", [C], {}),                ("ttm_trend", [H, L, C], {}),
    ("vortex", [H, L, C], {"length": 14}),
    # --- overlap / moving averages ------------------------------------------
    ("sma",  [C], {"length": 20}),          ("ema", [C], {"length": 200}),
    ("dema", [C], {"length": 20}),          ("tema", [C], {"length": 20}),
    ("trima", [C], {"length": 20}),         ("wma", [C], {"length": 20}),
    ("hma",  [C], {"length": 20}),          ("alma", [C], {"length": 20}),
    ("fwma", [C], {"length": 14}),          ("kama", [C], {"length": 14}),
    ("pwma", [C], {"length": 20}),          ("sinwma", [C], {"length": 20}),
    ("swma", [C], {"length": 20}),          ("t3", [C], {"length": 20}),
    ("vidya", [C], {"length": 14}),         ("zlma", [C], {"length": 20}),
    ("mcgd", [C], {"length": 14}),          ("hwma", [C], {}),
    ("rma",  [C], {"length": 14}),          ("ssf", [C], {"length": 10}),
    ("linreg", [C], {"length": 14}),        ("midpoint", [C], {"length": 14}),
    ("midprice", [H, L], {"length": 14}),   ("vwma", [C, V], {"length": 20}),
    ("hl2",  [H, L], {}),                   ("hlc3", [H, L, C], {}),
    ("ohlc4", [O, H, L, C], {}),            ("wcp", [H, L, C], {}),
    # --- volume ---------------------------------------------------------------
    ("ad",   [H, L, C, V], {}),             ("adosc", [H, L, C, V], {"fast": 3, "slow": 10}),
    ("aobv", [C, V], {}),                   ("cmf", [H, L, C, V], {}),
    ("efi",  [C, V], {"length": 13}),       ("eom", [H, L, C, V], {}),
    ("mfi",  [H, L, C, V], {"length": 14}), ("nvi", [C, V], {}),
    ("obv",  [C, V], {}),                   ("pvi", [C, V], {}),
    ("pvo",  [V], {}),                      ("pvol", [C, V], {}),
    ("pvr",  [C, V], {}),                   ("pvt", [C, V], {}),
    ("vwap", [H, L, C, V], {}),
    # --- statistics / returns --------------------------------------------------
    ("entropy", [C], {}),                   ("kurtosis", [C], {}),
    ("skew", [C], {}),                      ("mad", [C], {}),
    ("median", [C], {}),                    ("quantile", [C], {"length": 20}),
    ("zscore", [C], {}),                    ("slope", [C], {"length": 14}),
    ("log_return", [C], {}),                ("percent_return", [C], {}),
    ("decay", [C], {}),
    # --- candles ---------------------------------------------------------------
    ("cdl_doji", [O, H, L, C], {}),         ("cdl_inside", [O, H, L, C], {}),
    ("ha", [O, H, L, C], {}),
]


def run_specs(frame, specs):
    """Compute every spec. Return (columns_frame, failures)."""
    out, failures = {}, []
    for name, args, kwargs in specs:
        fn = getattr(ta, name, None)
        if fn is None:
            failures.append((name, "not present in this pandas_ta build"))
            continue
        try:
            res = fn(*[frame[a] for a in args], **kwargs)
        except Exception as exc:                      # reported, never swallowed
            failures.append((name, f"{type(exc).__name__}: {exc}"))
            continue
        if res is None:
            failures.append((name, "returned None"))
            continue
        if isinstance(res, pd.Series):
            out[res.name or name] = res
        else:
            for col in res.columns:
                out[col] = res[col]
    return pd.DataFrame(out, index=frame.index), failures


panel, failures = run_specs(df, SPECS)

print(f"specs run : {len(SPECS)}")
print(f"columns   : {panel.shape[1]}")
print(f"failures  : {len(failures)}")
for name, why in failures:
    print(f"   FAILED  {name}: {why}")
if not failures:
    print("\nEvery spec executed and returned columns. That is the assertion the original"
          "\nbare `except: continue` was unable to make.")

specs run : 117
columns   : 170
failures  : 0

Every spec executed and returned columns. That is the assertion the original
bare `except: continue` was unable to make.


### Warm-up is real, and it is not uniform

Every indicator with a window needs history before it can produce a number. That is not a bug, but
it *is* a trap if you slice the tail of a frame and assume the columns are populated. Here are the
hungriest columns in the panel.

In [5]:
warmup = panel.notna().idxmax()                     # first non-NaN position per column
warmup_bars = pd.Series({c: panel.index.get_loc(warmup[c]) for c in panel.columns})
allnan = [c for c in panel.columns if panel[c].isna().all()]

print(f"columns entirely NaN on this frame: {len(allnan)}  {allnan if allnan else ''}")
print(f"median warm-up: {int(warmup_bars.median())} bars    max: {int(warmup_bars.max())} bars\n")
print("Slowest ten to produce their first value:")
warmup_bars.sort_values(ascending=False).head(10).to_frame("first value at bar")

columns entirely NaN on this frame: 0  
median warm-up: 14 bars    max: 199 bars

Slowest ten to produce their first value:


,first value at bar
EMA_200,199
KSTs_9,52
KST_10_15_20_30_10_10_10_15,44
EBSW_40_10,39
AO_5_34,33
MASSI_9_25,32
INERTIA_20_14,32
STOCHRSId_14_14_3_3,31
DPO_20,30
VAR_30,29


## 4. Multi-output indicators

Several indicators return a set of related columns rather than one line — a long branch and a short
branch, or a level plus a direction. Raw, they are awkward. Consolidating them into one value plus
one state is usually what you actually wanted.

In [6]:
signals = pd.DataFrame(index=df.index)

# Parabolic SAR — long and short stops are mutually exclusive; merge into one level.
psar = ta.psar(df[H], df[L], df[C])
psar_long = psar.filter(like="PSARl").iloc[:, 0]
psar_short = psar.filter(like="PSARs").iloc[:, 0]
signals["PSAR"] = psar_long.combine_first(psar_short)
signals["PSAR_DIR"] = np.where(psar_long.notna(), 1, np.where(psar_short.notna(), -1, 0))

# Supertrend — keep the direction flag, which is the usable part.
st = ta.supertrend(df[H], df[L], df[C], length=10, multiplier=3)
signals["SUPERT_DIR"] = st["SUPERTd_10_3.0"]

# QQE — same long/short shape as PSAR.
qqe = ta.qqe(df[C])
signals["QQE"] = qqe.filter(regex=r"^QQE_\d").iloc[:, 0]
signals["QQE_DIR"] = np.where(qqe.filter(like="QQEl").iloc[:, 0].notna(), 1,
                       np.where(qqe.filter(like="QQEs").iloc[:, 0].notna(), -1, 0))

# TD Sequential — up-count and down-count merged to a signed count.
td = ta.td_seq(df[C], show_all=True)
td_up, td_dn = td["TD_SEQ_UPa"], td["TD_SEQ_DNa"]
signals["TD_SEQ"] = td_up.combine_first(-td_dn)

# Ichimoku returns a TUPLE: (history, forward projection). The second frame is the
# future-dated span, which is exactly why it must never be concatenated blindly.
ichi_hist, ichi_fwd = ta.ichimoku(df[H], df[L], df[C])
signals["TENKAN"] = ichi_hist["ITS_9"]
signals["KIJUN"] = ichi_hist["IKS_26"]

print(f"ichimoku history frame  : {ichi_hist.shape}")
print(f"ichimoku forward frame  : {ichi_fwd.shape}   <- projected past the last bar, not history")
signals.tail(3)

ichimoku history frame  : (753, 5)
ichimoku forward frame  : (26, 2)   <- projected past the last bar, not history


,PSAR,PSAR_DIR,SUPERT_DIR,QQE,QQE_DIR,TD_SEQ,TENKAN,KIJUN
date,,,,,,,,
2026-09-09,303.829686,1,1,53.186206,1,NaN,320.104996,315.690002
2026-09-10,304.908898,1,1,53.186206,1,NaN,320.354996,315.690002
2026-09-11,305.944942,1,1,53.186206,1,NaN,323.059998,318.395004


⚠ **`ichimoku` returns a tuple, and the second element is dated into the future.** It is the cloud
projection — legitimate for plotting, and a look-ahead leak if you concatenate it into a feature
frame. This is a real distinction the API makes and it is easy to miss.

## 5. What this fork adds

Everything above runs on upstream `pandas-ta` too. This section does not.

Three groups: TradingView built-ins that had no Python implementation, TA-Lib functions upstream
never wrapped, and price-structure work — chart patterns and Smart Money Concepts.

In [7]:
fork = pd.DataFrame(index=df.index)


def add(frame_or_series):
    if isinstance(frame_or_series, pd.Series):
        fork[frame_or_series.name] = frame_or_series
    else:
        for col in frame_or_series.columns:
            fork[col] = frame_or_series[col]


# --- TradingView built-ins ------------------------------------------------------
add(ta.pzo(df[C]))                     # Price Zone Oscillator
add(ta.szo(df[C]))                     # Sentiment Zone Oscillator
add(ta.vzo(df[C], df[V]))              # Volume Zone Oscillator
add(ta.wpo(df[H], df[C]))              # Wave Period Oscillator
add(ta.frama(df[H], df[L], df[C]))     # Fractal Adaptive MA
add(ta.ht(df[C]))                      # Hilbert Transform line
add(ta.stc_tv(df[C]))                  # Schaff Trend Cycle, TradingView variant
add(ta.vstop(df[H], df[L], df[C]))     # Volatility Stop
add(ta.vstop2(df[H], df[L], df[C]))

# --- TA-Lib gaps ----------------------------------------------------------------
add(ta.imi(df[O], df[C]))              # Intraday Momentum Index
add(ta.sarext(df[H], df[L], df[C]))    # Extended Parabolic SAR
add(ta.ht_trendline(df[C]))
add(ta.mama(df[C]))                    # MESA Adaptive MA + FAMA
add(ta.cvi(df[H], df[L]))              # Chaikin Volatility
add(ta.bw_mfi(df[H], df[L], df[C], df[V]))
add(ta.wilder_rma(df[C]))              # true Wilder smoothing

# --- chart patterns --------------------------------------------------------------
add(ta.head_shoulders(df[H], df[L], df[C]))
add(ta.triple_top_bottom(df[H], df[L], df[C]))
add(ta.triangle_wedge(df[H], df[L], df[C]))
add(ta.rounding_cup(df[C]))

# --- price structure / SMC --------------------------------------------------------
add(ta.bos(df[H], df[L], df[C]))       # break of structure
add(ta.choch(df[H], df[L], df[C]))     # change of character
add(ta.fvg(df[H], df[L], df[C]))       # fair value gaps
add(ta.ob(df[O], df[H], df[L], df[C])) # order blocks
add(ta.smc_sweep(df[O], df[H], df[L], df[C]))
add(ta.halftrend(df[H], df[L], df[C]))
add(ta.wavetrend(df[H], df[L], df[C]))
add(ta.zigzag(df[C]))
add(ta.linreg_channel(df[C]))
add(ta.ichimoku_ml(df[H], df[L], df[C]))   # 5 price lines -> 8 scale-free causal columns

print(f"{fork.shape[1]} columns from indicators this fork added.\n")
fired = fork.notna().sum()
print("Binary structure columns and how often they fire on this frame:")
for col in ["BOS_BULL", "BOS_BEAR", "CHoCH_BULL", "CHoCH_BEAR",
            "FVG_BULL", "FVG_BEAR", "IN_FVG_BULL", "IN_FVG_BEAR", "OB_BULL", "OB_BEAR"]:
    if col in fork:
        s = fork[col].dropna()
        rate = float((s > 0).mean()) * 100 if len(s) else float("nan")
        print(f"   {col:14s} {rate:6.2f}% of {len(s)} bars")

78 columns from indicators this fork added.

Binary structure columns and how often they fire on this frame:
   BOS_BULL        17.13% of 753 bars
   BOS_BEAR         5.31% of 753 bars
   CHoCH_BULL       1.46% of 753 bars
   CHoCH_BEAR       1.20% of 753 bars
   FVG_BULL        18.46% of 753 bars
   FVG_BEAR        14.08% of 753 bars
   IN_FVG_BULL     20.19% of 753 bars
   IN_FVG_BEAR     30.81% of 753 bars
   OB_BULL         18.33% of 753 bars
   OB_BEAR         17.66% of 753 bars


### Price levels are shipped as distances, on purpose

Look at the names: `FRAMA_DIST_PCT_16`, `VSTOP_DIST_PCT_20_1.0`, `HT_PCT`. A raw price level is
useless to a model — `SMA_10 = 187.4` means nothing without knowing what `close` was, and its
meaning drifts as the nominal price drifts over years.

So indicators whose natural output is a price level ship the **percent distance** instead, with the
raw level available behind `raw=True` for plotting. Section 6 shows why.

In [8]:
levels = ta.frama(df[H], df[L], df[C], raw=True)
dists = ta.frama(df[H], df[L], df[C])

print("raw=True  ->", list(levels.columns) if hasattr(levels, "columns") else levels.name)
print("default   ->", dists.name)
pd.DataFrame({"raw level": levels.iloc[:, 0] if hasattr(levels, "columns") else levels,
              "shipped distance %": dists}).tail(4)

raw=True  -> FRAMA_16
default   -> FRAMA_DIST_PCT_16


,raw level,shipped distance %
date,,
2026-09-08,314.661979,0.495142
2026-09-09,314.730011,0.193812
2026-09-10,315.452923,3.524166
2026-09-11,317.041893,4.803181


## 6. The ML feature contract, measured

Three properties. Each one is *tested* below on the columns this notebook has already computed —
not asserted in a docstring.

### 6a. Causal — bar `T` may only read bars `≤ T`

The test: take the frame, multiply everything from bar `cut` onward by 1.4, recompute, and compare
the first `cut` bars. If a single earlier value moved, the indicator read the future.

This probe is not decoration. It is how a genuine look-ahead bug was found in `ssf` in this
codebase: its recursion started at index 0, so `ssf.iloc[i-1]` evaluated `ssf.iloc[-1]` — the **last
bar of the series** — and the recursion carried that backwards contamination forward through every
subsequent bar. Perturbing only bars ≥ 300 of a 400-bar frame moved **bar 0 by 31.4**.

The exception list in the documentation had said the causality exceptions were "complete" at two
columns. They were three. A list contains only the cases someone already thought of; a probe finds
the ones nobody did.

In [9]:
def causality_report(frame, calls, cut=None):
    """Perturb the tail; any earlier bar that moves means the indicator read the future."""
    cut = cut or int(len(frame) * 0.6)
    bumped = frame.copy()
    for col in ("open", "high", "low", "close"):
        bumped.iloc[cut:, bumped.columns.get_loc(col)] *= 1.4

    rows = []
    for label, fn in calls.items():
        try:
            a, b = fn(frame), fn(bumped)
            a = a.to_frame() if isinstance(a, pd.Series) else a
            b = b.to_frame() if isinstance(b, pd.Series) else b
            for col in a.columns:
                x = np.asarray(a[col], dtype=float)[:cut]
                y = np.asarray(b[col], dtype=float)[:cut]
                m = np.isfinite(x) & np.isfinite(y)
                if not m.any():
                    continue
                drift = float(np.abs(x[m] - y[m]).max())
                rows.append({"column": col, "max prefix drift": drift,
                             "causal": "yes" if drift == 0.0 else "NO — READS THE FUTURE"})
        except Exception as exc:
            rows.append({"column": label, "max prefix drift": np.nan, "causal": f"error: {exc}"})
    return pd.DataFrame(rows).sort_values("max prefix drift", ascending=False)


checks = {
    "ssf":            lambda f: ta.ssf(f[C], length=10),
    "rsi":            lambda f: ta.rsi(f[C], length=14),
    "frama":          lambda f: ta.frama(f[H], f[L], f[C]),
    "vstop":          lambda f: ta.vstop(f[H], f[L], f[C]),
    "ichimoku_ml":    lambda f: ta.ichimoku_ml(f[H], f[L], f[C]),
    "bos":            lambda f: ta.bos(f[H], f[L], f[C]),
    "dpo(centered)":  lambda f: ta.dpo(f[C]),            # the DEFAULT — see the warning below
    "dpo(centered=False)": lambda f: ta.dpo(f[C], centered=False),
}

report = causality_report(df, checks)
leaks = report[report["causal"].str.startswith("NO")]
print(f"columns probed        : {len(report)}")
print(f"reading the future    : {len(leaks)}\n")
print(report.head(12).to_string(index=False))

columns probed        : 19
reading the future    : 1

                 column  max prefix drift                causal
                 DPO_20         46.104572 NO — READS THE FUTURE
               SSF_10_2          0.000000                   yes
       ICHI_CLOUD_THICK          0.000000                   yes
              LOWER_LOW          0.000000                   yes
            HIGHER_HIGH          0.000000                   yes
               BOS_BEAR          0.000000                   yes
               BOS_BULL          0.000000                   yes
   ICHI_CHIKOU_VS_CLOUD          0.000000                   yes
ICHI_FUTURE_CLOUD_COLOR          0.000000                   yes
       ICHI_CLOUD_COLOR          0.000000                   yes
                 RSI_14          0.000000                   yes
      ICHI_TK_CROSS_AGE          0.000000                   yes


⚠ **`dpo` ships `centered=True` as its default, and that default is a look-ahead.** It should appear
in the leak list above. This is inherited upstream behaviour, it is documented, and it is
deliberately *not* silently changed here — but `dpo` is registered in `Category`, which means a
blind `df.ta.strategy()` sweep will hand you a future-reading column and say nothing.

Pass `centered=False`. The row above it in the table shows that version is clean.

This is the honest version of a feature contract: the exceptions are named and demonstrated, not
buried.

### 6b. Scale-free — the column must not change when the price does

Multiply every price by 8. A ratio, a distance, or a bounded oscillator is unmoved. A price-scaled
column moves by a factor of 8 and is therefore not comparable across tickers, or across time on the
same ticker as its nominal price drifts.

The test reports **two** verdicts, and the gap between them matters. `bit-identical` is the strict
form. `scale-free` allows a floating-point tolerance, because a handful of columns are
scale-invariant in algebra yet differ in the final bit — a log return is the clean example.
Calling those failures would be wrong, so both numbers are shown.

Both kinds are legitimate outputs. The point is knowing which is which *before* handing them to a
model that cannot tell.

In [10]:
def scale_report(frame, columns_fn, k=8, tol=1e-9):
    """Two verdicts, because one is not enough.

    `bit-identical` is the strict test. `scale-free` allows a floating-point
    tolerance -- a log return is scale-invariant in algebra, yet
    log(k*a) - log(k*b) can differ from log(a) - log(b) in the last bit, and
    calling that a failure would be wrong.
    """
    scaled = frame.copy()
    for col in ("open", "high", "low", "close"):
        scaled[col] *= k
    a, b = columns_fn(frame), columns_fn(scaled)
    rows = []
    for col in a.columns:
        if col not in b:
            continue
        x, y = np.asarray(a[col], float), np.asarray(b[col], float)
        m = np.isfinite(x) & np.isfinite(y)
        if not m.any():
            continue
        drift = float(np.abs(x[m] - y[m]).max())
        scale = max(float(np.abs(x[m]).max()), 1e-12)
        rows.append({"column": col, "max drift": drift,
                     "bit-identical": drift == 0.0,
                     "scale-free": drift / scale < tol})
    return pd.DataFrame(rows)


# Names as pandas_ta actually emits them - ATR is ATRr_14 (RMA-smoothed), zscore is Z_30.
sample = ["RSI_14", "SMA_20", "STOCHk_14_3_3", "ATRr_14", "NATR_14",
          "OBV", "Z_30", "BBU_20_2.0", "HLC3", "LOGRET_1"]

scaled_panel = scale_report(df, lambda f: run_specs(f, SPECS)[0], k=8)
tot = len(scaled_panel)
free = int(scaled_panel["scale-free"].sum())
strict = int(scaled_panel["bit-identical"].sum())
print(f"panel columns : {tot}")
print(f"scale-free    : {free} ({free / tot:.1%})   of which bit-identical: {strict}")
print(f"price-scaled  : {tot - free}\n")

print("A few, named:")
print(scaled_panel[scaled_panel.column.isin(sample)].to_string(index=False))
print("\nPrice-scaled columns are not defects — they are levels. They are simply not features.")
print("ATRr_14 scales with price; NATR_14 is the same idea normalised, and does not.")
print("LOGRET_1 is scale-free but NOT bit-identical: log(k*a) - log(k*b) differs from")
print("log(a) - log(b) in the final floating-point bit. That is why the test has two columns.")

panel columns : 170
scale-free    : 93 (54.7%)   of which bit-identical: 87
price-scaled  : 77

A few, named:
       column    max drift  bit-identical  scale-free
      ATRr_14 8.072257e+01          False       False
      NATR_14 0.000000e+00           True        True
   BBU_20_2.0 2.423662e+03          False       False
       RSI_14 0.000000e+00           True        True
STOCHk_14_3_3 0.000000e+00           True        True
       SMA_20 2.268612e+03          False       False
         HLC3 2.378206e+03          False       False
          OBV 0.000000e+00           True        True
         Z_30 0.000000e+00           True        True
     LOGRET_1 8.881784e-16          False        True

Price-scaled columns are not defects — they are levels. They are simply not features.
ATRr_14 scales with price; NATR_14 is the same idea normalised, and does not.
LOGRET_1 is scale-free but NOT bit-identical: log(k*a) - log(k*b) differs from
log(a) - log(b) in the final floating-point bit. Tha

In [11]:
# The fork's own additions, held to the same test.
fork_scale = scale_report(df, lambda f: pd.DataFrame({
    "FRAMA_DIST_PCT_16":      ta.frama(f[H], f[L], f[C]),
    "HT_PCT":                 ta.ht(f[C]),
    "PZO_14":                 ta.pzo(f[C]),
    "VZO_14":                 ta.vzo(f[C], f[V]),
    "IMI_14":                 ta.imi(f[O], f[C]),
    **{c: ta.ichimoku_ml(f[H], f[L], f[C])[c]
       for c in ta.ichimoku_ml(f[H], f[L], f[C]).columns},
    **{c: ta.vstop(f[H], f[L], f[C])[c] for c in ["VSTOP_DIST_PCT_20_1.0", "VSTOP_TREND_20_1.0"]},
}), k=8)

print(f"fork columns tested : {len(fork_scale)}")
print(f"scale-free          : {int(fork_scale['scale-free'].sum())}"
      f"   (bit-identical: {int(fork_scale['bit-identical'].sum())})")
print(fork_scale.to_string(index=False))

fork columns tested : 15
scale-free          : 15   (bit-identical: 15)
                 column  max drift  bit-identical  scale-free
      FRAMA_DIST_PCT_16        0.0           True        True
                 HT_PCT        0.0           True        True
                 PZO_14        0.0           True        True
                 VZO_14        0.0           True        True
                 IMI_14        0.0           True        True
    ICHI_PRICE_VS_CLOUD        0.0           True        True
    ICHI_PRICE_VS_KIJUN        0.0           True        True
           ICHI_TK_DIST        0.0           True        True
      ICHI_TK_CROSS_AGE        0.0           True        True
       ICHI_CLOUD_COLOR        0.0           True        True
       ICHI_CLOUD_THICK        0.0           True        True
ICHI_FUTURE_CLOUD_COLOR        0.0           True        True
   ICHI_CHIKOU_VS_CLOUD        0.0           True        True
  VSTOP_DIST_PCT_20_1.0        0.0           True        Tru

`ichimoku_ml` is the reference conversion in this fork: Ichimoku's five *price lines* — all
price-scaled, none usable as features — become eight columns that are scale-free and causal, with
the same information content.

### 6c. Non-redundant — a column that restates another adds nothing

Two columns with a Spearman rank correlation near 1.0 are the same signal wearing different names.
Shipping both does not add information; it double-weights one signal.

In this fork the rule is enforced with a number, and the outcome is usually deletion: a candidate
measured at **ρ ≈ 0.90** against anything already shipped gets reverted rather than added.
`tvstop`'s distance column, `flag_breakout`'s pole measure and `range_profile`'s oscillator were all
built, measured, and deleted on exactly this test.

Here is the same test run across the panel this notebook just computed.

In [12]:
num = panel.select_dtypes("number")
num = num.loc[:, num.notna().sum() > 100]
corr = num.corr(method="spearman").abs()
np.fill_diagonal(corr.values, np.nan)

pairs = (corr.where(np.triu(np.ones(corr.shape), 1).astype(bool))
             .stack().sort_values(ascending=False))

print(f"columns compared      : {corr.shape[0]}")
print(f"pairs at |rho| >= 0.99: {int((pairs >= 0.99).sum())}")
print(f"pairs at |rho| >= 0.90: {int((pairs >= 0.90).sum())}\n")
print("Most redundant pairs in this panel:")
pairs.head(12).to_frame("|spearman rho|")

columns compared      : 170
pairs at |rho| >= 0.99: 595
pairs at |rho| >= 0.90: 1998

Most redundant pairs in this panel:


|spearman rho|
LOGRET_1    PCTRET_1           1.000000
HILOl_14_14 HILOs_14_14        1.000000
OHLC4       HA_close           1.000000
ACCBM_20    BBM_20_2.0         1.000000
            SMA_20             1.000000
HILO_14_14  HILOl_14_14        1.000000
            HILOs_14_14        1.000000
HLC3        VWAP_D             1.000000
BBM_20_2.0  SMA_20             1.000000
RSI_14      CMO_14             1.000000
HLC3        WCP                0.999961
WCP         VWAP_D             0.999961

Those near-duplicate pairs are the reason a feature set cannot just be "every indicator". Most of
what a naive sweep produces is the same handful of signals restated — and a model handed the same
signal ten times will weight it ten times.

One subtlety this fork learned the expensive way: **a redundancy gate measured only against what is
already shipped is blind to redundancy among the new candidates themselves.** Three columns can each
pass individually against the existing set and still be near-copies of one another.

## 7. Coverage, computed rather than claimed

The count below is read from the installed package at run time. It is not typed into this notebook,
because a number that is typed in is a number that goes stale.

In [13]:
registered = set()
for names in ta.Category.values():
    registered.update(names)

spec_names = {n for n, _, _ in SPECS}
# Names called OUTSIDE the SPECS table, in the cells above. `vp` and `macd` were listed here
# for a while and never actually called anywhere in the notebook, which inflated the number
# under a heading that says "computed rather than claimed". Anything added here must have a
# real call site in a cell above.
direct = {"psar", "qqe", "td_seq", "ichimoku", "supertrend",
          "pzo", "szo", "vzo", "wpo", "frama", "ht", "stc_tv", "vstop", "vstop2",
          "imi", "sarext", "ht_trendline", "mama", "cvi", "bw_mfi", "wilder_rma",
          "head_shoulders", "triple_top_bottom", "triangle_wedge", "rounding_cup",
          "bos", "choch", "fvg", "ob", "smc_sweep", "halftrend", "wavetrend",
          "zigzag", "linreg_channel", "ichimoku_ml", "rsi"}
used = spec_names | direct

print(f"indicators registered in Category : {len(registered)}")
print(f"exercised by this notebook        : {len(used & registered)}"
      f"  ({len(used & registered) / len(registered):.0%})")
print(f"not exercised here                : {len(registered - used)}")
print(f"total columns produced            : {panel.shape[1] + fork.shape[1] + signals.shape[1]}")

print("\nCoverage by category:")
for cat in sorted(ta.Category):
    names = set(ta.Category[cat])
    hit = len(names & used)
    bar = "#" * int(20 * hit / max(len(names), 1))
    print(f"  {cat:14s} {hit:3d}/{len(names):3d}  {bar}")

indicators registered in Category : 233
exercised by this notebook        : 148  (64%)
not exercised here                : 85
total columns produced            : 256

Coverage by category:
  candles          3/  5  ############
  cycles           2/  8  #####
  momentum        42/ 56  ###############
  overlap         35/ 49  ##############
  performance      2/  3  #############
  statistics       9/ 12  ###############
  trend           26/ 58  ########
  volatility      14/ 19  ##############
  volume          15/ 23  #############


Coverage is deliberately partial, and the fraction is stated rather than rounded up. Exercising
every registered name in one notebook would make it a test suite, not an example.

Note also what is **absent by design**: `beta` needs a benchmark series, and `up_and_down_volume`
and `volume_delta` need a lower-timeframe frame. All three are kept out of `Category` for that
reason, so a bulk sweep cannot call them with inputs it cannot supply.

## Where to go next

| You want to… | Do this |
|---|---|
| See every column this package can emit, with its observed form | `python docs/gen_indicator_dictionary.py` |
| Read one indicator's reference page | `docs/indicators/<name>.md` |
| Understand the rules for adding an indicator | the porting protocol in `CLAUDE.md` |
| Run everything at once | `df.ta.strategy()` — but read §6a first, `dpo` is in there |

**The one thing worth carrying away.** The indicators are the easy part; nearly every library has
them. What decides whether a column is usable is whether it is causal, whether it is scale-free, and
whether it is already in your feature set under another name — and none of those three is visible by
looking at the output. Each has to be measured, which is what §6 does and what the rest of this
repository is organised around.

A number nobody measured is a number nobody should trust — including the ones in this notebook,
every one of which was computed by the cell above it.